# Canonical Baseline Evaluation

This notebook supersedes the earlier baseline artifact. Every model pipeline embeds deterministic `ISUZU -> Isuzu` normalization, so training, evaluation, serialization, and future prediction use the same feature-quality rule.

In [1]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().resolve().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.preprocessing import build_preprocessor
from ml.preprocessing.brand_normalization import normalize_brand_labels

DATA_PATH = Path("../data/processed/cardekho_model_ready.csv")
MODEL_PATH = Path("../models/autoworth_random_forest_baseline.joblib")
PREPROCESSOR_PATH = Path("../models/preprocessor.joblib")
TARGET = "selling_price"
RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
X = df.drop(columns=TARGET)
y = df[TARGET].copy()
numerical_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

assert X.shape == (15_235, 11)
assert TARGET not in X.columns
assert {"brand", "model"}.issubset(categorical_features)


## Evaluation partitions

The ordinary random split is retained for comparison. The grouped split derives labels from the full canonical feature profile, so no identical profile can occur in both partitions. Preprocessing is still fitted only inside each training partition.

In [2]:
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

canonical_features = normalize_brand_labels(X)
profile_group_series = pd.Series(
    pd.MultiIndex.from_frame(canonical_features).factorize()[0], index=X.index
)
group_splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
group_train_idx, group_test_idx = next(group_splitter.split(X, y, groups=profile_group_series))
X_train_grouped, X_test_grouped = X.iloc[group_train_idx], X.iloc[group_test_idx]
y_train_grouped, y_test_grouped = y.iloc[group_train_idx], y.iloc[group_test_idx]

random_train_profiles = set(profile_group_series.loc[X_train_random.index])
random_test_profile_matches = int(profile_group_series.loc[X_test_random.index].isin(random_train_profiles).sum())
assert not set(profile_group_series.iloc[group_train_idx]) & set(profile_group_series.iloc[group_test_idx])

split_audit = pd.Series({
    "random_train_rows": len(X_train_random),
    "random_test_rows": len(X_test_random),
    "random_test_rows_matching_train_profiles": random_test_profile_matches,
    "grouped_train_rows": len(X_train_grouped),
    "grouped_test_rows": len(X_test_grouped),
    "grouped_shared_profiles": len(set(profile_group_series.iloc[group_train_idx]) & set(profile_group_series.iloc[group_test_idx])),
}, name="count")
split_audit.to_frame()

,count
random_train_rows,12188
random_test_rows,3047
random_test_rows_matching_train_profiles,390
grouped_train_rows,12201
grouped_test_rows,3034
grouped_shared_profiles,0


## Leakage-safe baseline comparison

`build_preprocessor` returns an unfitted pipeline. A new instance is fitted for every model and partition, so learned imputers, scalers, and encoders never cross from one evaluation partition to another.

In [3]:
estimators = {
    "Dummy Regressor": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
}

def evaluate_protocol(evaluation_name, X_train, y_train, X_test, y_test):
    rows = []
    for model_name, estimator in estimators.items():
        model_pipeline = Pipeline([
            ("preprocessor", build_preprocessor(numerical_features, categorical_features)),
            ("model", clone(estimator)),
        ])
        model_pipeline.fit(X_train, y_train)
        predictions = model_pipeline.predict(X_test)
        rows.append({
            "Evaluation": evaluation_name,
            "Model": model_name,
            "MAE": mean_absolute_error(y_test, predictions),
            "RMSE": mean_squared_error(y_test, predictions) ** 0.5,
            "R2": r2_score(y_test, predictions),
        })
    return pd.DataFrame(rows)

random_results = evaluate_protocol("Random split", X_train_random, y_train_random, X_test_random, y_test_random)
grouped_results = evaluate_protocol("Grouped split", X_train_grouped, y_train_grouped, X_test_grouped, y_test_grouped)
comparison_results = pd.concat([random_results, grouped_results], ignore_index=True)
comparison_results.sort_values(["Evaluation", "MAE"]).reset_index(drop=True)

,Evaluation,Model,MAE,RMSE,R2
0,Grouped split,Random Forest,99730.040382,2.053731e+05,0.941082
1,Grouped split,Linear Regression,179633.418057,3.658135e+05,0.813071
2,Grouped split,Ridge Regression,186295.089593,3.717045e+05,0.807002
3,Grouped split,Dummy Regressor,459322.972341,8.462681e+05,-0.000400
4,Random split,Random Forest,102039.353090,5.253224e+05,0.737043
5,Random split,Linear Regression,175013.713842,6.921704e+05,0.543481
6,Random split,Ridge Regression,177624.148053,6.772516e+05,0.562948
7,Random split,Dummy Regressor,446357.959120,1.024471e+06,-0.000075


## Save the canonical baseline artifact

The final model is fitted on all available cleaned rows only after evaluation. Its embedded preprocessing pipeline normalizes the brand label at prediction time. The standalone preprocessor artifact is overwritten with this same fitted canonical pipeline for compatibility with later project components.

In [4]:
final_model = Pipeline([
    ("preprocessor", build_preprocessor(numerical_features, categorical_features)),
    ("model", RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)),
])
final_model.fit(X, y)

fitted_preprocessor = final_model.named_steps["preprocessor"]
fitted_feature_names = fitted_preprocessor.get_feature_names_out().tolist()
assert "brand_Isuzu" in fitted_feature_names
assert "brand_ISUZU" not in fitted_feature_names

MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump(final_model, MODEL_PATH)
joblib.dump(fitted_preprocessor, PREPROCESSOR_PATH)

artifact_audit = pd.Series({
    "saved_feature_count": len(fitted_feature_names),
    "contains_canonical_isuzu": "brand_Isuzu" in fitted_feature_names,
    "contains_noncanonical_isuzu": "brand_ISUZU" in fitted_feature_names,
}, name="value")
artifact_audit.to_frame()

,value
saved_feature_count,167
contains_canonical_isuzu,True
contains_noncanonical_isuzu,False
